# 05 — Condition B2: Graph Neural Network (GNN)

**Project:** DDI-Graph-LLM  
**Phase 5:** Train a GNN (GCN) for edge-level interaction-type classification. Unlike Random Forest which uses hand-crafted features, the GNN learns node representations **end-to-end** directly from graph topology.

**Architecture:**
1. Node features (degree, centrality, etc.) → 2-layer GCN → node embeddings
2. For each edge (u, v): concatenate [h_u || h_v] → MLP → K classes

**Input:** `../data/edge_features.csv`  
**Evaluation:** Macro-F1 on the same 1,000-sample test subset


In [ ]:
import pandas as pd
import numpy as np
import time
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


## 1. Prepare Graph Data for PyTorch Geometric

In [ ]:
df = pd.read_csv("../data/edge_features.csv")
df_sample = pd.read_csv("../data/test_sample_1000.csv")

print(f"Full dataset: {len(df):,} edges")
print(f"Test subset: {len(df_sample):,} samples")

FEATURE_COLS = [
    "out_degree_u", "in_degree_u", "betweenness_u", "clustering_u", "pagerank_u",
    "out_degree_v", "in_degree_v", "betweenness_v", "clustering_v", "pagerank_v",
    "common_neighbors", "jaccard", "same_community", "degree_diff",
]


### 1.1 Map Drug Names to Integer IDs

PyTorch Geometric needs integer node IDs and a dense node feature matrix.


In [ ]:
# Get all unique drug names
all_drugs = sorted(set(df['drug_u'].unique()) | set(df['drug_v'].unique()))
drug_to_id = {drug: i for i, drug in enumerate(all_drugs)}
id_to_drug = {i: drug for drug, i in drug_to_id.items()}
num_nodes = len(all_drugs)
print(f"Number of nodes: {num_nodes}")

# Map edges to integer IDs
src_ids = df['drug_u'].map(drug_to_id).values
dst_ids = df['drug_v'].map(drug_to_id).values

# Edge index (2 x num_edges)
edge_index = torch.tensor(np.stack([src_ids, dst_ids]), dtype=torch.long)
print(f"Edge index shape: {edge_index.shape}")

# Edge labels
le = LabelEncoder()
edge_labels_all = le.fit_transform(df['label'].values)
edge_labels = torch.tensor(edge_labels_all, dtype=torch.long)
num_classes = len(le.classes_)
print(f"Classes ({num_classes}): {list(le.classes_)}")


### 1.2 Build Node Feature Matrix

Each node gets a 5-dimensional feature vector from its precomputed graph properties. We use the **source-side** features for each drug (averaging over all edges where it appears as source).


In [ ]:
# Compute node features by aggregating from edge features
node_feat_cols_u = ["out_degree_u", "in_degree_u", "betweenness_u", "clustering_u", "pagerank_u"]
node_feat_cols_v = ["out_degree_v", "in_degree_v", "betweenness_v", "clustering_v", "pagerank_v"]

# Build node feature matrix
node_features = np.zeros((num_nodes, 5))
node_counts = np.zeros(num_nodes)

# From source side
for i, row in df.iterrows():
    nid = drug_to_id[row['drug_u']]
    node_features[nid] += row[node_feat_cols_u].values.astype(float)
    node_counts[nid] += 1

# From target side
for i, row in df.iterrows():
    nid = drug_to_id[row['drug_v']]
    feats = row[node_feat_cols_v].values.astype(float)
    node_features[nid] += feats
    node_counts[nid] += 1

# Average
node_counts[node_counts == 0] = 1
node_features = node_features / node_counts[:, None]

# Normalize features to [0, 1]
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
node_features = scaler.fit_transform(node_features)

x = torch.tensor(node_features, dtype=torch.float)
print(f"Node feature matrix shape: {x.shape}")
print(f"Feature stats: min={x.min():.3f}, max={x.max():.3f}, mean={x.mean():.3f}")


### 1.3 Train/Test Edge Split

We use the same test samples as other conditions. All remaining edges are used for training.


In [ ]:
# Identify test edges by matching drug pairs
test_pairs = set(zip(df_sample['drug_u'], df_sample['drug_v']))

test_mask = np.array([
    (row['drug_u'], row['drug_v']) in test_pairs 
    for _, row in df.iterrows()
])
train_mask = ~test_mask

print(f"Train edges: {train_mask.sum():,}")
print(f"Test edges: {test_mask.sum():,}")

train_mask_t = torch.tensor(train_mask, dtype=torch.bool)
test_mask_t = torch.tensor(test_mask, dtype=torch.bool)


In [ ]:
# Create PyTorch Geometric Data object
data = Data(
    x=x,
    edge_index=edge_index,
    edge_attr=edge_labels,
    num_nodes=num_nodes,
)
data = data.to(device)

print(f"PyG Data object:")
print(f"  Nodes: {data.num_nodes}")
print(f"  Edges: {data.num_edges}")
print(f"  Node features: {data.x.shape}")
print(f"  Edge labels: {data.edge_attr.shape}")


## 2. Define GNN Model

**Architecture:**
- 2-layer GCN to learn node embeddings from graph structure
- For each edge (u, v): concatenate node embeddings [h_u || h_v]
- MLP classifier: concat → hidden → ReLU → output (K classes)


In [ ]:
class GCNEdgeClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        # GCN layers for node embedding
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        
        # MLP for edge classification
        # Input: concatenation of two node embeddings [h_u || h_v]
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_channels, num_classes),
        )
    
    def forward(self, x, edge_index):
        # Node embedding via GCN
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        h = self.conv2(h, edge_index)
        
        # Edge classification: concat source and target embeddings
        src, dst = edge_index
        edge_repr = torch.cat([h[src], h[dst]], dim=1)
        
        return self.edge_mlp(edge_repr)


model = GCNEdgeClassifier(
    in_channels=5,          # 5 node features
    hidden_channels=64,     # hidden dimension
    num_classes=num_classes, # 7 interaction types
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")


## 3. Train the GNN

In [ ]:
# Class weights for imbalanced labels
class_counts = Counter(edge_labels_all[train_mask])
total_train = train_mask.sum()
class_weights = torch.tensor(
    [total_train / (num_classes * class_counts[i]) for i in range(num_classes)],
    dtype=torch.float
).to(device)
print(f"Class weights: {class_weights.cpu().numpy().round(2)}")

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 200
train_losses = []
test_f1s = []

t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask_t], data.edge_attr[train_mask_t])
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    
    # Evaluate every 20 epochs
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            pred = out[test_mask_t].argmax(dim=1).cpu().numpy()
            true = data.edge_attr[test_mask_t].cpu().numpy()
            f1 = f1_score(true, pred, average='macro', zero_division=0)
            test_f1s.append((epoch, f1))
            elapsed = time.time() - t0
            print(f"  Epoch {epoch:>3d} | Loss: {loss.item():.4f} | Test Macro-F1: {f1:.4f} | {elapsed:.1f}s")

print(f"\nTraining complete in {time.time()-t0:.1f}s")


### 3.1 Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(train_losses, color='#e74c3c', alpha=0.7)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss')

# F1 curve
epochs_eval, f1_eval = zip(*test_f1s)
axes[1].plot(epochs_eval, f1_eval, marker='o', color='#2ecc71')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro-F1')
axes[1].set_title('Test Macro-F1 During Training')
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()


## 4. Final Evaluation

In [ ]:
model.eval()
with torch.no_grad():
    out = model(data.x, data.edge_index)
    pred_all = out.argmax(dim=1).cpu().numpy()

# Evaluate on test mask
y_true = data.edge_attr[test_mask_t].cpu().numpy()
y_pred = pred_all[test_mask_t]

# Convert back to label names
y_true_labels = le.inverse_transform(y_true)
y_pred_labels = le.inverse_transform(y_pred)

gnn_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
gnn_acc = np.mean(y_true == y_pred)

print("=" * 50)
print("CONDITION B2: GNN (GCN) RESULTS")
print("=" * 50)
print(f"Accuracy:     {gnn_acc:.4f}")
print(f"Macro-F1:     {gnn_f1:.4f}")

print(f"\n--- Per-Class Report ---")
print(classification_report(y_true_labels, y_pred_labels, zero_division=0))


### 4.1 Confusion Matrix

In [ ]:
labels = sorted(le.classes_)
fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(y_true_labels, y_pred_labels, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap='Purples', values_format='d')
ax.set_title('Condition B2: GNN (GCN) — Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 5. Evaluate on Same 1000-Sample Subset

For fair comparison with conditions A, B1, and C, compute metrics on the exact same test samples.


In [ ]:
# Map sample drug pairs to edge indices
sample_preds = []
sample_true = []

for _, row in df_sample.iterrows():
    # Find this edge in the full dataset
    mask = (df['drug_u'] == row['drug_u']) & (df['drug_v'] == row['drug_v'])
    idx = df.index[mask]
    if len(idx) > 0:
        edge_idx = idx[0]
        sample_preds.append(le.inverse_transform([pred_all[edge_idx]])[0])
        sample_true.append(row['label'])

sample_preds = np.array(sample_preds)
sample_true = np.array(sample_true)

gnn_f1_subset = f1_score(sample_true, sample_preds, average='macro', zero_division=0)
gnn_acc_subset = np.mean(sample_true == sample_preds)

print(f"GNN on 1000-sample subset:")
print(f"  Macro-F1:  {gnn_f1_subset:.4f}")
print(f"  Accuracy:  {gnn_acc_subset:.4f}")


## 6. Four-Way Comparison

In [ ]:
# Known results from other phases
llm_f1 = 0.2185       # Phase 4
agent_f1 = 0.2321     # Phase 6 (update if re-run)

# RF on same subset
from sklearn.ensemble import RandomForestClassifier
train_mask_rf = ~df.index.isin(df_sample.index)
X_train_rf = df.loc[train_mask_rf, FEATURE_COLS].values
y_train_rf = df.loc[train_mask_rf, 'label'].values

clf = RandomForestClassifier(
    n_estimators=200, max_depth=None, random_state=42,
    class_weight="balanced", n_jobs=-1
)
clf.fit(X_train_rf, y_train_rf)
rf_preds = clf.predict(df_sample[FEATURE_COLS].values)
rf_f1 = f1_score(df_sample['label'].values, rf_preds, average='macro')

print("=" * 60)
print("FOUR-WAY COMPARISON (on same 1000-sample subset)")
print("=" * 60)
print(f"{'Condition':<30} {'Macro-F1':>10}")
print("-" * 42)
print(f"{'Random baseline (1/7)':<30} {1/7:>10.4f}")
print(f"{'A: LLM-only':<30} {llm_f1:>10.4f}")
print(f"{'C: LangGraph Agent':<30} {agent_f1:>10.4f}")
print(f"{'B2: GNN (GCN)':<30} {gnn_f1_subset:>10.4f}")
print(f"{'B1: Random Forest':<30} {rf_f1:>10.4f}")
print("-" * 42)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

conditions = ['Random\n(1/K)', 'A: LLM\nonly', 'C: LangGraph\nAgent', 'B2: GNN\n(GCN)', 'B1: Random\nForest']
scores = [1/7, llm_f1, agent_f1, gnn_f1_subset, rf_f1]
colors = ['#95a5a6', '#e67e22', '#2ecc71', '#9b59b6', '#3498db']

bars = ax.bar(conditions, scores, color=colors, edgecolor='white', linewidth=1.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Macro-F1 Score', fontsize=12)
ax.set_title('DDI Type Prediction: Four-Way Comparison', fontsize=14)
ax.set_ylim(0, 1.15)
ax.axhline(y=1/7, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("../data/four_way_comparison.png", dpi=150, bbox_inches='tight')
plt.show()


## 7. Save Results

In [ ]:
# Save GNN predictions on test subset
df_sample['pred_gnn'] = sample_preds
df_sample.to_csv("../data/results_condition_b2.csv", index=False)
print(f"Saved results_condition_b2.csv")

# Save model
torch.save(model.state_dict(), "../data/gnn_model.pt")
print(f"Saved gnn_model.pt")

print(f"\nFinal: Condition B2 (GNN) Macro-F1 = {gnn_f1_subset:.4f}")


## Summary

**Condition B2 (GNN) complete.**

| Condition | Macro-F1 | Method |
|-----------|----------|--------|
| Random baseline | 0.1429 | 1/K guess |
| A: LLM-only | 0.2185 | GPT-4o-mini, drug names only |
| C: LangGraph Agent | ~0.23 | GPT-4o-mini + graph context |
| **B2: GNN (GCN)** | **TBD** | **End-to-end graph learning** |
| B1: Random Forest | ~0.93 | Hand-crafted graph features |

**Key question:** Does the GNN, which learns representations end-to-end from graph structure, approach the RF's performance? Or does hand-crafted feature engineering still win?

**Next:** Final evaluation notebook aggregating all results.
